# Public research notebook

This notebook is a cleaned public version of the original
research workflow.

## Execution model

- All filesystem paths are relative to the repository root.
- No external mounted filesystem is required.
- Stored cell outputs have been removed.
- Generated files are written below the local `results/`
  directory.
- The archival source notebook remains unchanged.


In [ ]:
# Portable repository configuration
#
# The notebook assumes that it is executed from the repository
# root or from a cloned copy of the repository.

from pathlib import Path

REPOSITORY_ROOT = Path.cwd().resolve()

# Move upward when the notebook is launched from a nested folder.
if REPOSITORY_ROOT.name in {
    "lorenz",
    "rossler",
    "duffing",
    "kuramoto",
    "stuart_landau",
    "coupled_map_lattice",
}:
    REPOSITORY_ROOT = REPOSITORY_ROOT.parents[1]

DATA_DIR = REPOSITORY_ROOT / "data"
RESULTS_DIR = REPOSITORY_ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
TABLES_DIR = RESULTS_DIR / "tables"
CHECKPOINTS_DIR = RESULTS_DIR / "checkpoints"

for directory in [
    DATA_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    TABLES_DIR,
    CHECKPOINTS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {REPOSITORY_ROOT}")
print(f"Results directory: {RESULTS_DIR}")


In [ ]:
# ============================================================
# DELTA WINDOW PROJECT — ROSSLER PIPELINE INIT
# ============================================================


import os
import shutil
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

PROJECT_NAME = "delta_window_rossler_v2"
BASE_DIR = "data/raw"

PROJECT_DIR = f"{BASE_DIR}/{PROJECT_NAME}"
CSV_DIR = f"{PROJECT_DIR}/csv"
FIG_DIR = f"{PROJECT_DIR}/figures"
FINAL_FIG_DIR = f"{PROJECT_DIR}/final_figures"
CHECKPOINT_DIR = f"{PROJECT_DIR}/checkpoints"
LOG_DIR = f"{PROJECT_DIR}/logs"

for d in [PROJECT_DIR, CSV_DIR, FIG_DIR, FINAL_FIG_DIR, CHECKPOINT_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

def save_csv(df, name, folder=CSV_DIR):
    path = f"{folder}/{name}_{TIMESTAMP}.csv"
    df.to_csv(path, index=False)
    print(f"[CSV SAVED] {path}")
    return path

def save_checkpoint(df, name):
    path = f"{CHECKPOINT_DIR}/{name}_CHECKPOINT.csv"
    df.to_csv(path, index=False)
    print(f"[CHECKPOINT SAVED] {path}")
    return path

def save_figure(plt_obj, name, folder=FIG_DIR):
    path = f"{folder}/{name}_{TIMESTAMP}.png"
    plt_obj.savefig(path, dpi=300, bbox_inches="tight")
    print(f"[FIGURE SAVED] {path}")
    return path

def save_final_figure(plt_obj, name):
    path = f"{FINAL_FIG_DIR}/{name}_{TIMESTAMP}.png"
    plt_obj.savefig(path, dpi=300, bbox_inches="tight")
    print(f"[FINAL FIGURE SAVED] {path}")
    return path

with open(f"{LOG_DIR}/session_info_{TIMESTAMP}.json", "w") as f:
    json.dump(
        {"project_name": PROJECT_NAME, "timestamp": TIMESTAMP},
        f,
        indent=4
    )

print("\n===================================================")
print("ROSSLER PIPELINE INITIALIZED")
print(PROJECT_DIR)
print("===================================================")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =========================
# parameters RÖSSLERA
# =========================

a = 0.2
b = 0.2
c = 5.7

DT = 0.01
STEPS = 15000
RUNS = 10

delta_values = np.linspace(0.1, 3.0, 30)

# =========================
# SYMULACJA Z FILTREM Δ
# =========================

def simulate_rossler(delta, damping, rng):
    x = 0.1 + 0.01 * rng.normal()
    y = 0.1 + 0.01 * rng.normal()
    z = 0.1 + 0.01 * rng.normal()

    xs, ys, zs = [], [], []
    transitions = 0

    prev_sign = np.sign(x)

    for _ in range(STEPS):
        dx = -y - z
        dy = x + a*y
        dz = b + z*(x - c)

        # surowy krok
        new_x = x + dx * DT
        new_y = y + dy * DT
        new_z = z + dz * DT

        # length step
        jump = np.sqrt((new_x-x)**2 + (new_y-y)**2 + (new_z-z)**2)

        # filtr Δ
        if jump > delta:
            scale = delta / jump
            new_x = x + (new_x - x) * scale * damping
            new_y = y + (new_y - y) * scale * damping
            new_z = z + (new_z - z) * scale * damping

        x, y, z = new_x, new_y, new_z

        xs.append(x)
        ys.append(y)
        zs.append(z)

        sign = np.sign(x)
        if sign != 0 and prev_sign != 0 and sign != prev_sign:
            transitions += 1
        if sign != 0:
            prev_sign = sign

    return transitions, np.array(xs), np.array(ys), np.array(zs)

# =========================
# DWELL TIME
# =========================

def get_dwell_times(xs):
    xs = np.array(xs, dtype=float)
    signs = np.sign(xs)

    # Pad with zeros
    for i in range(1, len(signs)):
        if signs[i] == 0:
            signs[i] = signs[i-1]

    dwell = []
    count = 1

    for i in range(1, len(signs)):
        if signs[i] == signs[i-1]:
            count += 1
        else:
            dwell.append(count)
            count = 1

    if count > 0:
        dwell.append(count)

    return np.array(dwell)

# =========================
# EXPLORATION (occupancy)
# =========================

def occupancy(xs, zs, bins=60):
    H, _, _ = np.histogram2d(xs, zs, bins=bins)
    return np.sum(H > 0)

In [ ]:
DAMPING = 0.5

mean_transitions = []
mean_dwell = []
mean_occupancy = []

for delta in delta_values:
    t_runs = []
    d_runs = []
    o_runs = []

    for run in range(RUNS):
        rng = np.random.default_rng(run)

        t, xs, ys, zs = simulate_rossler(delta, DAMPING, rng)

        dwell = get_dwell_times(xs)
        mean_d = np.mean(dwell) * DT if len(dwell) > 0 else np.nan
        occ = occupancy(xs, zs)

        t_runs.append(t)
        d_runs.append(mean_d)
        o_runs.append(occ)

    mean_transitions.append(np.mean(t_runs))
    mean_dwell.append(np.mean(d_runs))
    mean_occupancy.append(np.mean(o_runs))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.plot(delta_values, mean_transitions, 'o-')
plt.title("Transitions vs Delta (Rössler)")
plt.grid()
plt.show()

plt.figure(figsize=(8,5))
plt.plot(delta_values, mean_dwell, 'o-')
plt.title("Dwell vs Delta (Rössler)")
plt.grid()
plt.show()

plt.figure(figsize=(8,5))
plt.plot(delta_values, mean_occupancy, 'o-')
plt.title("Exploration vs Delta (Rössler)")
plt.grid()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =========================
# PHASE-BASED STATE DEFINITION FOR THE RÖSSLER SYSTEM
# =========================

def phase_regions(xs, ys):
    """
    Regiony na podstawie kąta fazowego phi = atan2(y, x).
    Zwraca znaki regionów:
    +1 dla phi >= 0
    -1 dla phi < 0
    """
    phi = np.arctan2(ys, xs)
    regions = np.where(phi >= 0, 1, -1)
    return phi, regions

def count_phase_transitions(xs, ys):
    _, regions = phase_regions(xs, ys)
    return np.sum(regions[1:] != regions[:-1])

def get_phase_dwell_times(xs, ys):
    _, regions = phase_regions(xs, ys)

    dwell = []
    count = 1

    for i in range(1, len(regions)):
        if regions[i] == regions[i-1]:
            count += 1
        else:
            dwell.append(count)
            count = 1

    dwell.append(count)
    return np.array(dwell, dtype=float)

In [ ]:
DAMPING = 0.5

mean_phase_transitions = []
mean_phase_dwell = []
mean_phase_occupancy = []

for delta in delta_values:
    t_runs = []
    d_runs = []
    o_runs = []

    for run in range(RUNS):
        rng = np.random.default_rng(run)

        t_old, xs, ys, zs = simulate_rossler(delta, DAMPING, rng)

        # Transitions and dwell times computed from phase states
        t_phase = count_phase_transitions(xs, ys)
        dwell_phase = get_phase_dwell_times(xs, ys)
        mean_d = np.mean(dwell_phase) * DT if len(dwell_phase) > 0 else np.nan

        occ = occupancy(xs, zs)

        t_runs.append(t_phase)
        d_runs.append(mean_d)
        o_runs.append(occ)

    mean_phase_transitions.append(np.mean(t_runs))
    mean_phase_dwell.append(np.mean(d_runs))
    mean_phase_occupancy.append(np.mean(o_runs))

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(delta_values, mean_phase_transitions, 'o-')
plt.title("Phase transitions vs Delta (Rössler)")
plt.xlabel("Delta")
plt.ylabel("Mean phase transitions")
plt.grid()
plt.show()

plt.figure(figsize=(8,5))
plt.plot(delta_values, mean_phase_dwell, 'o-')
plt.title("Phase dwell vs Delta (Rössler)")
plt.xlabel("Delta")
plt.ylabel("Mean phase dwell time")
plt.grid()
plt.show()

plt.figure(figsize=(8,5))
plt.plot(delta_values, mean_phase_occupancy, 'o-')
plt.title("Exploration vs Delta (Rössler, phase states)")
plt.xlabel("Delta")
plt.ylabel("Mean occupancy")
plt.grid()
plt.show()

In [ ]:
np.savez("rossler_data.npz",
         delta_norm=np.array(delta_norm, dtype=float),
         exploration=np.array(mean_occupancy, dtype=float),
         filtered=np.array(mean_filtered, dtype=float))

print("saved rossler_data.npz")

In [ ]:
# ============================================================
# FINAL UNIVERSAL SAVE BLOCK — ROSSLER
# ============================================================

import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

FINAL_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

print("\n===================================================")
print("STARTING FINAL AUTO SAVE — ROSSLER")
print("===================================================")

# ============================================================
# SAVE ALL DATAFRAMES
# ============================================================

saved_dfs = []

for var_name, var_value in list(globals().items()):

    if isinstance(var_value, pd.DataFrame):

        try:
            save_path = f"{CSV_DIR}/{var_name}_{FINAL_TIMESTAMP}.csv"
            var_value.to_csv(save_path, index=False)

            saved_dfs.append({
                "name": var_name,
                "rows": len(var_value),
                "columns": len(var_value.columns),
                "path": save_path
            })

            print(f"[DATAFRAME SAVED] {var_name}")
            print(save_path)

        except Exception as e:
            print(f"[ERROR SAVING {var_name}] {e}")

index_df = pd.DataFrame(saved_dfs)
index_path = f"{CSV_DIR}/RECOVERED_DATAFRAME_INDEX_{FINAL_TIMESTAMP}.csv"
index_df.to_csv(index_path, index=False)

print("\n[DATAFRAME INDEX SAVED]")
print(index_path)

# ============================================================
# CREATE ROSSLER SUMMARY TABLES FROM LISTS
# ============================================================

try:
    if "delta_values" in globals():

        n = len(delta_values)

        summary_dict = {
            "delta": np.array(delta_values, dtype=float)
        }

        possible_cols = {
            "mean_transitions": "mean_transitions",
            "mean_dwell": "mean_dwell",
            "mean_occupancy": "mean_occupancy",
            "mean_phase_transitions": "mean_phase_transitions",
            "mean_phase_dwell": "mean_phase_dwell",
            "mean_phase_occupancy": "mean_phase_occupancy",
        }

        for var, col in possible_cols.items():
            if var in globals() and len(globals()[var]) == n:
                summary_dict[col] = np.array(globals()[var], dtype=float)
                print(f"[SUMMARY COLUMN ADDED] {var}")

        rossler_summary_df = pd.DataFrame(summary_dict)

        summary_path = f"{CSV_DIR}/rossler_summary_recovered_{FINAL_TIMESTAMP}.csv"
        rossler_summary_df.to_csv(summary_path, index=False)

        print("\n[ROSSLER SUMMARY SAVED]")
        print(summary_path)

except Exception as e:
    print("[ROSSLER SUMMARY ERROR]")
    print(e)

# ============================================================
# SAVE IMPORTANT ROSSLER ARRAYS/LISTS AS NPZ
# ============================================================

array_names = [
    "delta_values",
    "mean_transitions",
    "mean_dwell",
    "mean_occupancy",
    "mean_phase_transitions",
    "mean_phase_dwell",
    "mean_phase_occupancy",
    "delta_norm",
    "mean_filtered",
]

saved_arrays = {}

for name in array_names:
    if name in globals():
        try:
            saved_arrays[name] = np.array(globals()[name], dtype=float)
            print(f"[ARRAY FOUND] {name}, shape={saved_arrays[name].shape}")
        except Exception as e:
            print(f"[ARRAY ERROR] {name}: {e}")

if len(saved_arrays) > 0:
    npz_path = f"{CSV_DIR}/rossler_recovered_arrays_{FINAL_TIMESTAMP}.npz"
    np.savez(npz_path, **saved_arrays)
    print("\n[ARRAY NPZ SAVED]")
    print(npz_path)

# ============================================================
# SAFE rossler_data.npz EXPORT
# ============================================================

try:
    if all(x in globals() for x in ["delta_norm", "mean_occupancy", "mean_filtered"]):

        npz_path = f"{CSV_DIR}/rossler_data_{FINAL_TIMESTAMP}.npz"

        np.savez(
            npz_path,
            delta_norm=np.array(delta_norm, dtype=float),
            exploration=np.array(mean_occupancy, dtype=float),
            filtered=np.array(mean_filtered, dtype=float)
        )

        print("\n[ROSSLER DATA NPZ SAVED]")
        print(npz_path)

    else:
        print("\n[SKIPPED rossler_data.npz]")
        print("Missing one of: delta_norm, mean_occupancy, mean_filtered")

except Exception as e:
    print("[ROSSLER NPZ ERROR]")
    print(e)

# ============================================================
# SAVE ALL OPEN FIGURES
# ============================================================

fig_nums = plt.get_fignums()
saved_figs = []

print(f"\nOpen figures found: {len(fig_nums)}")

for i, fig_num in enumerate(fig_nums, start=1):

    try:
        fig = plt.figure(fig_num)
        fig_path = f"{FIG_DIR}/rossler_recovered_figure_{i}_{FINAL_TIMESTAMP}.png"
        fig.savefig(fig_path, dpi=300, bbox_inches="tight")

        saved_figs.append(fig_path)

        print(f"[FIGURE SAVED]")
        print(fig_path)

    except Exception as e:
        print(f"[ERROR SAVING FIGURE {i}] {e}")

# ============================================================
# ZIP BACKUP
# ============================================================

try:
    zip_path = f"{BASE_DIR}/{PROJECT_NAME}_FULL_BACKUP_{FINAL_TIMESTAMP}"

    shutil.make_archive(
        zip_path,
        "zip",
        PROJECT_DIR
    )

    print("\n[FULL ZIP BACKUP CREATED]")
    print(f"{zip_path}.zip")

except Exception as e:
    print("[ZIP BACKUP ERROR]")
    print(e)

print("\n===================================================")
print("FINAL SAVE COMPLETED — ROSSLER")
print(f"DataFrames saved: {len(saved_dfs)}")
print(f"Figures saved: {len(saved_figs)}")
print("===================================================")